In [1]:
!nvidia-smi
!pwd

Wed May  6 18:00:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install datasets sentence-transformers scikit-learn anthropic transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 753.6/753.6 kB 18.9 MB/s eta 0:00:00


In [3]:
from google.colab import userdata
import os
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

In [4]:
import os
os.makedirs("runs", exist_ok=True)
os.makedirs("cache", exist_ok=True)

In [5]:
# Install Ollama
!sudo apt-get install zstd
!curl -fsSL https://ollama.com/install.sh | sh

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 42 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (8,700 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 122354 files and directories currently

In [19]:
import subprocess
import time
import requests

# Start server
subprocess.Popen(["ollama", "serve"])

# Wait until it actually responds, not just a fixed sleep
print("Waiting for Ollama server...")
for i in range(30):
    try:
        r = requests.get("http://localhost:11434", timeout=2)
        print(f"Ollama is up after {i+1}s")
        break
    except Exception:
        time.sleep(1)
else:
    print("ERROR: Ollama did not start after 30s")

Waiting for Ollama server...
Ollama is up after 1s


In [20]:
# Pull the Gemma 2 9B (or 2B/4B depending on availability)
# Note: Gemma 2 9B fits well on a T4, but 2B is even faster.
%%shell
ollama serve &
sleep 5
ollama pull gemma2:2b
ollama list

Error: listen tcp 127.0.0.1:11434: bind: address already in use

NAME         ID              SIZE      MODIFIED               
gemma2:2b    8ccf136fdd52    1.6 GB    Less than a second ago    


In [ ]:
# Execute English baseline on the dev split[cite: 1]
!python exp0_baseline.py --split dev --lang en


  EXP 0 — Baseline | split=dev lang=en
  Loading queries for language: en...
README.md: 4.84kB [00:00, 9.86MB/s]
en_train.json: 4.00MB [00:00, 12.4MB/s]
en_dev.json: 1.05MB [00:00, 6.54MB/s]
Generating train split: 100% 14977/14977 [00:00<00:00, 98662.15 examples/s]
Generating dev split: 100% 3905/3905 [00:00<00:00, 139431.65 examples/s]
  Loading document collection...
collection_data.json: 100% 36.5M/36.5M [00:01<00:00, 33.2MB/s]
Generating collection split: 100% 10000/10000 [00:01<00:00, 8895.58 examples/s]
Loaded 3905 queries, 10000 documents
Loading model: intfloat/multilingual-e5-large
modules.json: 100% 387/387 [00:00<00:00, 1.88MB/s]
README.md: 160kB [00:00, 42.0MB/s]
sentence_bert_config.json: 100% 57.0/57.0 [00:00<00:00, 219kB/s]
config.json: 100% 690/690 [00:00<00:00, 2.63MB/s]
model.safetensors: 100% 2.24G/2.24G [00:13<00:00, 162MB/s]
Loading weights: 100% 391/391 [00:00<00:00, 901.04it/s, Materializing param=pooler.dense.weight]
XLMRobertaModel LOAD REPORT from: intfloat/

In [21]:
# Uses LLM to rewrite tweets as scientific claims[cite: 2]
%%shell
python exp1_tweet_normalization.py --split dev --lang en --provider ollama --ollama-model gemma2:2b


  EXP 1 — Tweet Normalization | split=dev lang=en provider=ollama
  Loading queries for language: en...
  Loading document collection...
Loaded 3905 queries, 10000 documents

Normalizing tweets (cache has 0 entries)...
  Normalizing [1/3905] qid=3 ...
  Normalizing [2/3905] qid=4 ...
  Normalizing [3/3905] qid=11 ...
  Normalizing [4/3905] qid=15 ...
  Normalizing [5/3905] qid=18 ...
  Normalizing [6/3905] qid=20 ...
  Normalizing [7/3905] qid=23 ...
  Normalizing [8/3905] qid=27 ...
  Normalizing [9/3905] qid=43 ...
  Normalizing [10/3905] qid=51 ...
  Normalizing [11/3905] qid=57 ...
  Normalizing [12/3905] qid=65 ...
  Normalizing [13/3905] qid=66 ...
  Normalizing [14/3905] qid=71 ...
  Normalizing [15/3905] qid=74 ...
  Normalizing [16/3905] qid=75 ...
  Normalizing [17/3905] qid=78 ...
  Normalizing [18/3905] qid=79 ...
  Normalizing [19/3905] qid=82 ...
  Normalizing [20/3905] qid=84 ...
  Normalizing [21/3905] qid=88 ...
  Normalizing [22/3905] qid=90 ...
  Normalizing [23/390

In [ ]:
# Execute Experiment 2 using the local Gemma model
!python exp2_claim_extraction.py --split dev --lang en --provider ollama --ollama-model gemma2:2b


  EXP 2 — Claim Extraction | split=dev lang=en provider=ollama
  Loading queries for language: en...
  Loading document collection...
Loaded 3905 queries, 10000 documents

Extracting claims (cache has 0 entries)...
  Extracting [1/3905] qid=3 ...
  Extracting [2/3905] qid=4 ...
  Extracting [3/3905] qid=11 ...
  Extracting [4/3905] qid=15 ...
  Extracting [5/3905] qid=18 ...
  Extracting [6/3905] qid=20 ...
  Extracting [7/3905] qid=23 ...
  Extracting [8/3905] qid=27 ...
  Extracting [9/3905] qid=43 ...
  Extracting [10/3905] qid=51 ...
  Extracting [11/3905] qid=57 ...
  Extracting [12/3905] qid=65 ...
  Extracting [13/3905] qid=66 ...
  Extracting [14/3905] qid=71 ...
  Extracting [15/3905] qid=74 ...
  Extracting [16/3905] qid=75 ...
  Extracting [17/3905] qid=78 ...
  Extracting [18/3905] qid=79 ...
  Extracting [19/3905] qid=82 ...
  Extracting [20/3905] qid=84 ...
  Extracting [21/3905] qid=88 ...
  Extracting [22/3905] qid=90 ...
  Extracting [23/3905] qid=95 ...
  Extracting 

In [22]:
# Translates German tweets to English before retrieval[cite: 4]
!python exp3_translation.py --lang de --provider local


  EXP 3 — Translation DE→EN | split=dev provider=local
  Loading queries for language: de...
de_train.json: 413kB [00:00, 24.0MB/s]
de_dev.json: 111kB [00:00, 9.23MB/s]
Generating train split: 100% 1460/1460 [00:00<00:00, 27508.57 examples/s]
Generating dev split: 100% 386/386 [00:00<00:00, 48675.67 examples/s]
  Loading document collection...
Loaded 386 queries, 10000 documents

Translating queries de→en (cache has 0 entries)...
  Loading translation model: Helsinki-NLP/opus-mt-de-en
tokenizer_config.json: 100% 42.0/42.0 [00:00<00:00, 175kB/s]
source.spm: 100% 797k/797k [00:00<00:00, 51.0MB/s]
target.spm: 100% 768k/768k [00:00<00:00, 39.0MB/s]
vocab.json: 1.27MB [00:00, 26.6MB/s]
/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
config.json: 1.38kB [00:00, 3.88MB/s]
pytorch_model.bin: 100% 298M/298M [00:02<00:00, 122MB/s]
Loading weigh